In [1]:
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
import jax
import jax.numpy as jnp
import pickle, sys, os, copy
sys.path.append('../../data_preprocessing')
sys.path.append('../../avgflow')
from utils.rdkit_utils import mirror_3d_mol
from preprocess import mol2features

In [3]:
toy_smiles = [
    'COc1ccc(S(=O)(=O)n2nc(OC(=O)c3ccccc3)cc2N)cc1',
    'Cc1nn(Cc2ccccc2)c(Cl)c1C(=O)OCC(=O)NC(=O)NC12CC3CC(CC(C3)C1)C2',
    'Cc1noc(C(=O)Nc2c(C)nn(Cc3ccccc3)c2C)c1Cl',
    'Cc1ccc(NCC(=O)N/N=C\\c2ccc([N+](=O)[O-])o2)cc1',
    'COc1cc(Cl)c(C)cc1NC(=O)CSc1nnc(-c2ccco2)n1CC(C)C',
    'Cc1ccc2c(C)cc(=S)[nH]c2c1',
    'O=C(OC[C@@H]1CCC[NH+]2CCCC[C@H]12)c1cc2ccccc2oc1=O',
    'COc1ccc(OC)c(NC(=O)CSc2nnc3ccc(-c4ccncc4)nn23)c1',
    'O=C(Cc1ccc(Cl)cc1)Nc1cccc(S(=O)(=O)Nc2ccccc2F)c1',
    'CCC(Sc1nc(C)cc(C)n1)C(=O)Nc1ccc(Cl)cn1',
    'Nc1ccc2nn(-c3ccc(Cl)cc3)nc2c1',
    'CC(C)Sc1nnc(NC(=O)c2ccco2)s1',
    'Cc1ccc(C)c(NC(=S)NCCCN2CCOCC2)c1',
    'O=c1[nH]cnc2c1ncn2Cc1ccc([N+](=O)[O-])cc1',
    'O=C(OC(C(=O)Nc1cc(Cl)cc(Cl)c1)c1ccccc1)C1=COCCO1',
    'C#CCNC(=O)C1=C[C@@H](c2ccc(Br)cc2)C[C@@H](OCc2ccc(CO)cc2)O1'
]

In [ ]:
# Create 8 random conformers for each SMILES
toy_mols = {}
for smi in toy_smiles:  
    # Get RDKit mol object from SMILES and add hydrogen atoms
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    
    # Generate conformers using RDKit, and optimize using MMFF
    AllChem.EmbedMultipleConfs(mol, numConfs=8, randomSeed=42)
    gen_confs = []
    for conf_id in range(mol.GetNumConformers()):
        AllChem.MMFFOptimizeMolecule(mol, confId=conf_id)
        gen_confs.append(np.array(mol.GetConformer(conf_id).GetPositions()))

    # Featurize the molecule
    features = mol2features(mol, 'drugs')

    # Save the processed molecule with conformers
    toy_mols[smi] = {
        'features': features,
        'conformers': gen_confs,
        'smiles': smi,
        'rdk_mol_with_confs': mol,
    }

pickle.dump(toy_mols, open('toy_mols.pkl', 'wb'))

In [ ]:
# Create 8 random conformers for each SMILES
toy_mols = {}
for smi in toy_smiles:  
    # Get RDKit mol object from SMILES and add hydrogen atoms
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)

    # For reflow, we need (X0', X1') pairs. Using random initial positions for X0's in toy example
    x0s = [np.random.randn(mol.GetNumAtoms(), 3) for _ in range(8)]

    # For toy reflow example, generate X1's using RDKit, and optimize using MMFF
    # For real reflow/distill finetuning, X1's should be generated by the model from X0's
    AllChem.EmbedMultipleConfs(mol, numConfs=8, randomSeed=42)
    gen_confs = []
    for conf_id in range(mol.GetNumConformers()):
        AllChem.MMFFOptimizeMolecule(mol, confId=conf_id)
        gen_confs.append(np.array(mol.GetConformer(conf_id).GetPositions()))


    # Featurize the molecule
    features = mol2features(mol, 'drugs')
    # Save the processed molecule with conformers
    toy_mols[smi] = {
        'features': features,
        'x0s': x0s,
        'x1s': gen_confs,
        'smiles': smi,
        'rdk_mol_with_confs': mol,
    }
pickle.dump(toy_mols, open('toy_mols_reflow.pkl', 'wb'))